In [1]:
import random
import os
import json
from tqdm import tqdm
from datetime import datetime
import datasets

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to the directory containing the test/train/validate_sample.csv obtained at the step before
sample_dir = '/projects/iris/rferreir/GeoBenchmark/data/NY-POI'

# Output dir
output_dir = '/projects/iris/rferreir/GeoBenchmark/data/NY-POI'

# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

## Get data from previous checkpoints 

In [3]:
import pandas as pd

test = set(pd.read_csv("/projects/iris/rferreir/datasets/NY/test_sample.csv")['trajectory_id'])
train = set(pd.read_csv("/projects/iris/rferreir/datasets/NY/train_sample.csv")['trajectory_id'])
print(len(test.intersection(train)))

0


In [4]:
print(len(test))
print(len(pd.read_csv("/projects/iris/rferreir/datasets/NY/test_sample.csv")['trajectory_id']))

1347
1347


In [5]:
all_data = pd.read_csv("/projects/iris/rferreir/datasets/NY/sample.csv")

In [6]:
test_data = pd.read_csv("/projects/iris/rferreir/datasets/NY/test_sample.csv")
columns = test_data.columns
new_trajs = [test_data]

def get_traj(row):
    traj_id = row['trajectory_id']
    df = all_data[all_data['trajectory_id'] == traj_id]
    df = df[columns]
    new_trajs.append(df)

test_data.apply(get_traj, axis=1)

print(len(new_trajs))
    

1348


In [7]:
df_final = pd.concat(new_trajs, ignore_index=True)

## Sort by time 

In [8]:
df_final['UTCTimeOffset'] = pd.to_datetime(df_final['UTCTimeOffset'], utc=True)  
df_final = df_final.sort_values(by='UTCTimeOffset', ascending=True)

In [9]:
df_final.to_csv(os.path.join(sample_dir, "test_sample2.csv"), index=False)

## Creating the dataset

In [14]:
# Code from LLMMove : main.py (https://github.com/LLMMove/LLMMove/blob/main/main.py)
def readTrain(filePath):
    longs = dict()
    pois = dict()
    with open(filePath, 'r') as file:
        lines = file.readlines()
    for line in lines[1:]:
        data = line.split(',')
        time, u, lati, longi, i, category = data[1], data[5], data[6], data[7], data[8], data[10]
        # Conversion vers le format ISO standard avec "Z" pour UTC
        time = datetime.fromisoformat(time)
        time = time.isoformat().replace("+00:00", "Z")
        if i not in pois:
            pois[i] = {"latitude": lati, "longitude": longi, "category": category}
        if u not in longs:
            longs[u] = list()
        longs[u].append((i, time))
    return longs, pois

def readTest(filePath):
    recents = dict()
    pois = dict()
    targets = dict()
    traj2u = dict()
    with open(filePath, 'r') as file:
        lines = file.readlines()
    for line in lines[1:]:
        data = line.split(',')
        time, trajectory, u, lati, longi, i, category = data[1], data[3],data[5], data[6], data[7], data[8], data[10]
        # Conversion vers le format ISO standard avec "Z" pour UTC
        time = datetime.fromisoformat(time)
        time = time.isoformat().replace("+00:00", "Z")
        if i not in pois:
            pois[i] = dict()
            pois[i]["latitude"] = lati
            pois[i]["longitude"] = longi
            pois[i]["category"] = category
        if trajectory not in traj2u:
            traj2u[trajectory] = u
        if trajectory not in recents:
            recents[trajectory] = list()
            recents[trajectory].append((i, time))
        else:
            if trajectory in targets:
                recents[trajectory].append(targets[trajectory])
            targets[trajectory] = (i, time)
    return recents, pois, targets, traj2u

def getData(datasetName='nyc'):
    if datasetName == 'nyc':
        filePath = os.path.join(sample_dir, '{}_sample{}.csv')
    else:
        raise NotImplementedError
    trainPath = filePath.format('train', '')
    testPath = filePath.format('test', '2')

    longs, poiInfos = readTrain(trainPath)
    print(len(longs))
    recents, testPoi, targets, traj2u = readTest(testPath)
    print(len(traj2u))
    poiInfos.update(testPoi)

    targets = dict(list(targets.items()))

    return longs, recents, targets, poiInfos, traj2u

In [22]:
# Code from LLMMove : LLMMove.py (https://github.com/LLMMove/LLMMove/blob/main/models/LLMMove.py)
from math import radians, sin, cos, sqrt, atan2
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1 = eval(lat1)
    lon1 = eval(lon1)
    lat2 = eval(lat2)
    lon2 = eval(lon2)
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    radius = 6371.0
    distance = radius * c
    return distance
    

class LLMMove():
    def run(self, data, datasetName):
        self.datasetName = datasetName
        self.longs, self.recents, self.targets, self.poiInfos, self.traj2u = data
        self.data = []
        poiList = list(self.poiInfos.keys())
        hit1 = 0
        hit5 = 0
        hit10 = 0
        rr = 0
        err = list()
        for trajectory, groundTruth in tqdm(self.targets.items()):
            seed_value = eval(trajectory)
            random.seed(seed_value)
            negSample = random.sample(poiList, 99)
            candidateSet = negSample + [groundTruth[0]]
            random.shuffle(candidateSet)
            
            self.runeach(trajectory, candidateSet, groundTruth)
    
    def runeach(self, trajectory, candidateSet, groundTruth):
        u = self.traj2u[trajectory]
        long = self.longs[u]
        rec = self.recents[trajectory]
        
        output = dict()
        assert rec[-1][0] == groundTruth[0]
        assert groundTruth[0] in candidateSet
        longterm = [(poi, self.poiInfos[poi]["category"], t) for poi, t in long]
        longterm = longterm[-40:]

        recent = [(poi, self.poiInfos[poi]["category"], t) for poi, t in rec][:-1]
        recent = recent[-5:]
        mostrec = recent[-1][0]
        
        candidates = [(poi, float(haversine_distance(self.poiInfos[poi]["latitude"], self.poiInfos[poi]["longitude"], self.poiInfos[mostrec]["latitude"], self.poiInfos[mostrec]["longitude"])), self.poiInfos[poi]["category"]) for poi in candidateSet]
        candidates.sort(key=lambda x:x[1])
        candidates = [(c[0], str(c[1]), c[2]) for c in candidates]

        dic = {
            "long-term_check-ins": longterm,
            "recent_check-ins":recent,
            "candidates":candidates,
            "ground_truth":[groundTruth[0]]
        }
        self.data.append(dic)
        
    def save(self):
        with open(os.path.join(output_dir, 'data.json'), 'w') as f:
            json.dump(self.data, f)
        
    

In [23]:
data = getData('nyc')

model = LLMMove()

model.run(data, 'nyc')
model.save()

1047
1347


100%|██████████| 1347/1347 [00:04<00:00, 324.69it/s]


## Converting to HF

In [24]:
d1 = datasets.load_dataset('json', data_files={"test":os.path.join(output_dir, 'data.json')})

print(d1['test'])

d1.save_to_disk(os.path.join(output_path, "NY-POI"))

Generating test split: 1347 examples [00:00, 4420.69 examples/s]


Dataset({
    features: ['long-term_check-ins', 'recent_check-ins', 'candidates', 'ground_truth'],
    num_rows: 1347
})


Saving the dataset (1/1 shards): 100%|██████████| 1347/1347 [00:00<00:00, 24159.00 examples/s]


## Sanity Check

In [26]:
import hashlib

d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", 'NY-POI')
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

assert content_hash(d1['test']) == content_hash(d2['test'])

Generating test split: 100%|██████████| 1347/1347 [00:00<00:00, 9885.23 examples/s] 
